In [1]:
import glob
import os
import copy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

pd.set_option('display.max_columns', None) # Displays all columns of df

In [141]:
# Load all files from one folder

Files_dir = '/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Link5'
Files_path = os.path.join(Files_dir,'*.csv') 
Files = glob.glob(Files_path)

In [142]:
Files

['/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Link5/100tp_561-100-50ms-1000g_24_conf561_merged_spots_tracks.csv',
 '/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Link5/100tp_561-100-50ms-1000g_4_conf561_merged_spots_tracks.csv',
 '/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Link5/100tp_561-100-50ms-1000g_9_conf561_merged_spots_tracks.csv',
 '/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Link5/100tp_561-100-50ms-1000g_19_conf561_merged_spots_tracks.csv',
 '/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Link5/100tp_561-100-50ms-1000g_18_conf561_merged_spots_tracks.csv',
 '/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Link5/100tp_561-100-50ms-1000g_15_conf561_merged_spots_tracks.csv',
 '/Users/abamford/

In [143]:
# Read csv files into list

all_files = []

for file in Files:
    table = pd.read_csv(file)
    all_files.append(table)

all_files[0].head()

,frame,roi_id,x,y,bg_denoised,amp_denoised,sigma_y,sigma_x,mean_spot_intensity,particle,track_id,unique_id
0,0,1.0,397.5,355.5,6730,9913,1,1,1205.638889,0,1.0_0.0,14
1,0,1.0,299.5,404.5,12897,21370,1,1,2219.166667,1,1.0_1.0,126
2,0,1.0,373.5,434.5,2757,6923,1,1,711.722222,5,1.0_5.0,34
3,0,1.0,417.5,436.5,4756,8864,1,1,876.805556,7,1.0_7.0,110
4,0,1.0,308.5,405.5,6558,16944,1,1,1826.444444,8,1.0_8.0,10


In [144]:
# Adds columns "line", "day" and "batch" to table

for file in all_files:
    
    # The strings need to be changed accordingly!

    line = "TUBB2B-KI"
    day = 21
    batch = "20230223"
   
    file["line"] = line
    file["day"] = day
    file["batch"] = batch
  

all_files[0].head()

,frame,roi_id,x,y,bg_denoised,amp_denoised,sigma_y,sigma_x,mean_spot_intensity,particle,track_id,unique_id,line,day,batch
0,0,1.0,397.5,355.5,6730,9913,1,1,1205.638889,0,1.0_0.0,14,TUBB2B-KI,21,20230223
1,0,1.0,299.5,404.5,12897,21370,1,1,2219.166667,1,1.0_1.0,126,TUBB2B-KI,21,20230223
2,0,1.0,373.5,434.5,2757,6923,1,1,711.722222,5,1.0_5.0,34,TUBB2B-KI,21,20230223
3,0,1.0,417.5,436.5,4756,8864,1,1,876.805556,7,1.0_7.0,110,TUBB2B-KI,21,20230223
4,0,1.0,308.5,405.5,6558,16944,1,1,1826.444444,8,1.0_8.0,10,TUBB2B-KI,21,20230223


In [145]:
# Adds column "file_name"

# List of file names within the folder
file_names = os.listdir(Files_dir)

for df, file_name in zip (all_files, file_names):
    df['file_name'] = file_name

all_files[1].head()

,frame,roi_id,x,y,bg_denoised,amp_denoised,sigma_y,sigma_x,mean_spot_intensity,particle,track_id,unique_id,line,day,batch,file_name
0,0,1.0,265.5,597.5,5636,11084,1,1,1158.250000,2,1.0_2.0,80,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_4_conf561_merged_spot...
1,0,1.0,276.5,597.5,6172,13250,1,1,1260.388889,3,1.0_3.0,141,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_4_conf561_merged_spot...
2,0,1.0,291.5,618.5,6157,11685,1,1,1260.250000,4,1.0_4.0,104,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_4_conf561_merged_spot...
3,0,1.0,273.5,603.5,6106,12398,1,1,1288.861111,5,1.0_5.0,215,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_4_conf561_merged_spot...
4,0,1.0,299.5,613.5,5847,9952,1,1,1178.555556,6,1.0_6.0,58,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_4_conf561_merged_spot...


In [146]:
def create_unique_track_id(row: pd.Series, file_identifier: str) -> str:
    """
    Function to create unique track-ids for all tracks of all ROIs per image
    by combining roi_id, particle number, and file information.
    """
    return f"{file_identifier}_{row['roi_id']}_{row['particle']}"

In [147]:
# Adds column containing a unique identifier base on file name, ROI and particle

all_files_clean = []

for i, df in enumerate(all_files):

    position = 11

    # Apply unique track IDs using the create_track_id function and and remove track_id column (redundant)
    df["roi_id"] = df["roi_id"].astype(int)
    df.insert(position, 'file_roi_particle', df.apply(lambda row: create_unique_track_id(row, i), axis=1))
    df = df.drop(columns=['track_id'])
    df = df.sort_values(by=["file_roi_particle", "frame"])

    all_files_clean.append(df)

all_files_clean[5].head()

,frame,roi_id,x,y,bg_denoised,amp_denoised,sigma_y,sigma_x,mean_spot_intensity,particle,file_roi_particle,unique_id,line,day,batch,file_name
0,0,1,476.5,532.5,6755,10043,1,1,1057.361111,2,5_1_2,129,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_15_conf561_merged_spo...
1,0,1,482.5,568.5,4905,12891,1,1,1256.972222,3,5_1_3,134,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_15_conf561_merged_spo...
2,0,1,494.5,573.5,5406,7474,1,1,775.750000,5,5_1_5,120,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_15_conf561_merged_spo...
3,1,1,443.5,428.5,4883,7966,1,1,909.861111,6,5_1_6,20,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_15_conf561_merged_spo...
4,1,1,476.5,532.5,6701,9116,1,1,1003.194444,2,5_1_2,129,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_15_conf561_merged_spo...


In [148]:
# Output directory

output_dir = r'/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Clean_files'


In [149]:
# Save clean tables

for table, csv_file in zip(all_files_clean, Files):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(csv_file))[0]
    print(file_name)
    
    # Define the output file path
    output_file = os.path.join(output_dir, file_name + '_clean.csv')
    print(output_file)

    # Save clean table
    table.to_csv(output_file, index=False)

100tp_561-100-50ms-1000g_24_conf561_merged_spots_tracks
/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Clean_files/100tp_561-100-50ms-1000g_24_conf561_merged_spots_tracks_clean.csv
100tp_561-100-50ms-1000g_4_conf561_merged_spots_tracks
/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Clean_files/100tp_561-100-50ms-1000g_4_conf561_merged_spots_tracks_clean.csv
100tp_561-100-50ms-1000g_9_conf561_merged_spots_tracks
/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Clean_files/100tp_561-100-50ms-1000g_9_conf561_merged_spots_tracks_clean.csv
100tp_561-100-50ms-1000g_19_conf561_merged_spots_tracks
/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Clean_files/100tp_561-100-50ms-1000g_19_conf561_merged_spots_tracks_clean.csv
100tp_561-100-50ms-1000g_18_conf561_merged_spots_tracks
/Users/abamford/Desktop/TUBB2B-KI_pr

In [150]:
# To merge all tracks belonging to same timepoint into one list while maintaining unique identity of every track

merged_file = pd.concat(all_files_clean).sort_values(by=["file_roi_particle", "frame"])
merged_file.head(100)


,frame,roi_id,x,y,bg_denoised,amp_denoised,sigma_y,sigma_x,mean_spot_intensity,particle,file_roi_particle,unique_id,line,day,batch,file_name
0,0,1,397.5,355.5,6730,9913,1,1,1205.638889,0,0_1_0,14,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_24_conf561_merged_spo...
21,1,1,398.5,355.5,6806,12232,1,1,1420.305556,0,0_1_0,14,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_24_conf561_merged_spo...
31,2,1,399.5,357.5,6238,10501,1,1,1144.416667,0,0_1_0,14,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_24_conf561_merged_spo...
48,3,1,398.5,357.5,7260,14780,1,1,1511.027778,0,0_1_0,14,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_24_conf561_merged_spo...
55,4,1,397.5,356.5,7470,11962,1,1,1265.138889,0,0_1_0,14,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_24_conf561_merged_spo...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1042,73,1,307.5,400.5,4306,11490,1,1,1036.388889,108,0_1_108,124,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_24_conf561_merged_spo...
1057,74,1,306.5,400.5,6226,12626,1,1,1429.000000,108,0_1_108,124,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_24_conf561_merged_spo...
1067,75,1,307.5,402.5,5324,12576,1,1,1245.833333,108,0_1_108,124,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_24_conf561_merged_spo...
1082,76,1,307.5,404.5,6907,12993,1,1,1375.861111,108,0_1_108,124,TUBB2B-KI,21,20230223,100tp_561-100-50ms-1000g_24_conf561_merged_spo...


In [152]:
# Save merged file

# Define the output file path
output_file = os.path.join(output_dir, 'Batch20230223_D21_50ms.csv')
print(output_file)

# Save merged table
merged_file.to_csv(output_file, index=False)

/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20240219/Clean_files/Batch20230223_D21_50ms.csv
